Imports for feature engineering

In [69]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import joblib

Load the cleaned up data set version 

In [70]:
df = pd.read_csv("../datasets/bank_cleaned.csv")

print("Cleaned dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

Cleaned dataset shape: (41176, 20)
Columns: ['age', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'campaign', 'pdays', 'previous', 'poutcome', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'y']


Separating the X and y value features

In [71]:
X = df.drop(columns=["y"])
y = df["y"]

print("Features shape:", X.shape)
print("Target shape:",y.shape)

Features shape: (41176, 19)
Target shape: (41176,)


Compartementalise the Numeric and Categorical columns apart

In [72]:
num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()

print("Numeric columns:", num_cols)
print("Categorical columns:", cat_cols)
                                                 

Numeric columns: ['age', 'campaign', 'pdays', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']
Categorical columns: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'poutcome']


C:\Users\Katleho\AppData\Local\Temp\ipykernel_33244\2151397681.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include=["object"]).columns.tolist()


Using OneHot encoding for the categorical values and Scaling for numerics

In [73]:
# ---Fix: use the OneHotEncoder instead of Label Encoder ---
#LabelEncoding would lead to an artificial order on the categories such as  <0<1, which would not right for nominal varials like job, martial, education and so on.
#OneHotEndcoding ensures the separation of the binary column for each category

# ---Fix: scale the  numeric features since logistic regression is sensitive to feature scale, so standerising numeric values columns

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y
)

X_train_processed =preprocessed =preprocess.fit_transform(X_train)
X_test_processed = preprocess.transform(X_test)
 

#Then get the feature names after one hot encoding
one = preprocess.named_transformers_["cat"]
cat_feature_names =one.get_feature_names_out(cat_cols).tolist()
feature_names = num_cols + cat_feature_names

print("Processed features shape:", X_train_processed.shape)
print("First few feature names", feature_names[:10])

Processed features shape: (32940, 62)
First few feature names ['age', 'campaign', 'pdays', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'job_admin.']


Training the test split using stratified  to have the same split for both models

In [74]:
# ---Fix: Use stratify=y tp preserve the class balance concern ---
# There is an imbalance of 89% no and 11% yes so we can use startify the split

X_train, X_test, y_train, y_test = train_test_split(
    X_processed, 
    y, 
    test_size=0.2, 
    random_state=42,
    stratify=y)
print("Train set shape:", X_train.shape)
print("Test set shape:", X_test.shape)
print("Train target distribution:\n", y_train.value_counts(normalize=True))
print("Test target distribution:\n", y_test.value_counts(normalize=True))

Train set shape: (32940, 62)
Test set shape: (8236, 62)
Train target distribution:
 y
no     0.887341
yes    0.112659
Name: proportion, dtype: float64
Test target distribution:
 y
no     0.887324
yes    0.112676
Name: proportion, dtype: float64


In [75]:
joblib.dump(
    (X_train_processed, X_test_processed, y_train, y_test),
    "../datasets/train_test_split.pkl" )

joblib.dump(preprocess, "../datasets/preprocessor.pkl")

print("Data preparation complete")
print("Processed data saved to datasets/train_test_split.pkl")
print("Preprocessor saved to datasets/preprocessor.pkl")

Data preparation complete
Processed data saved to datasets/train_test_split.pkl
Preprocessor saved to datasets/preprocessor.pkl
